In [1]:
"""
FAST RAG Baseline Implementation (Lewis et al., 2020)
"""

# ============================================================================
# INSTALL & IMPORTS (30 seconds)
# ============================================================================
print("Installing dependencies...")
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers", "transformers", "torch", "wikipedia-api", "wikipedia"])

import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from typing import List, Tuple, Dict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!\n")

Installing dependencies...
Setup complete!



In [2]:
# ============================================================================
# FAST CONFIGURATION
# ============================================================================
class Config:
    """OPTIMIZED for speed"""
    QUERIES_FILE = '/content/t2t_test_participants.jsonl'
    CORPUS_FILE = 'corpus.jsonl'
    OUTPUT_FILE = 'rag_results.jsonl'

    # FASTER MODELS (smaller but still good quality)
    EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # 6x faster than mpnet
    GENERATOR_MODEL = 'google/flan-t5-base'  # 3x faster than large, 300MB vs 3GB

    # REDUCED RETRIEVAL
    TOP_K = 3  # Retrieve fewer passages (faster generation)
    MAX_INPUT_LENGTH = 512  # Shorter context (faster generation)
    MAX_OUTPUT_LENGTH = 128  # Shorter answers (faster generation)

print("Fast configuration loaded!\n")

Fast configuration loaded!



In [3]:
# ============================================================================
# UPLOAD DATASET (LOCAL FILE)
# ============================================================================
import os

print("=" * 80)
print("STEP 1: LOAD DATASET")
print("=" * 80)
print(f"\nUsing queries file: {Config.QUERIES_FILE}")

if not os.path.exists(Config.QUERIES_FILE):
    raise FileNotFoundError(f"Queries file not found at {Config.QUERIES_FILE}")

with open(Config.QUERIES_FILE, 'r') as f:
    num_queries = sum(1 for _ in f)
print(f"\n Found queries file. {num_queries} queries found\n")

STEP 1: LOAD DATASET

Using queries file: /content/t2t_test_participants.jsonl

 Found queries file. 299 queries found



In [4]:
# ============================================================================
# FAST CORPUS CREATION
# ============================================================================
print("=" * 80)
print("STEP 2: CREATING CORPUS (FAST MODE)")
print("=" * 80)
print("\nCreating smaller corpus for speed...\n")

def create_fast_corpus():
    """Create smaller corpus - only 10 topics but more focused"""
    import wikipedia

    # REDUCED: Only most relevant topics
    topics = [
        "Four color theorem",  # For map coloring queries
        "Sunburn",            # For skin/sunburn queries
        "Cloud",              # For weather/cloud queries
        "Convolution",        # For math/signal processing
        "Stomach",            # For digestion queries
        "Aluminum",           # For material queries
        "Density",            # For physics queries
        "Weather",            # General weather
        "Mathematics",        # General math
        "Physics"             # General physics
    ]

    corpus = []
    for topic in tqdm(topics, desc="Fetching (fast)"):
        try:
            results = wikipedia.search(topic, results=1)
            if not results:
                continue
            page = wikipedia.page(results[0], auto_suggest=False)

            # Take only first 20 paragraphs per topic
            paragraphs = page.content.split('\n\n')[:20]

            for para in paragraphs:
                para = para.strip()
                if len(para) > 50 and not para.startswith('=='):
                    corpus.append({'text': para, 'source': page.title})
        except:
            continue

    with open(Config.CORPUS_FILE, 'w') as f:
        for doc in corpus:
            f.write(json.dumps(doc) + '\n')

    return corpus

try:
    corpus = [json.loads(line) for line in open(Config.CORPUS_FILE)]
    print(f" Loaded {len(corpus)} passages\n")
except:
    corpus = create_fast_corpus()
    print(f"\n Created {len(corpus)} passages\n")

STEP 2: CREATING CORPUS (FAST MODE)

Creating smaller corpus for speed...

 Loaded 104 passages



In [5]:
# ============================================================================
# FAST RETRIEVER
# ============================================================================
print("=" * 80)
print("STEP 3: DENSE RETRIEVER")
print("=" * 80)

class FastRetriever:
    """Faster retriever with smaller model"""
    def __init__(self, corpus):
        print(f"\nLoading FAST embedding model: {Config.EMBEDDING_MODEL}")
        self.model = SentenceTransformer(Config.EMBEDDING_MODEL)
        self.texts = [d['text'] for d in corpus]

        print(f"Encoding {len(self.texts)} passages (fast mode)...\n")
        self.embeddings = self.model.encode(
            self.texts,
            convert_to_tensor=True,
            show_progress_bar=True,
            batch_size=64  # INCREASED batch size for speed
        )
        print(f"\n Retriever ready! ({len(self.texts)} passages)\n")

    def retrieve(self, query, top_k=Config.TOP_K):
        q_emb = self.model.encode(query, convert_to_tensor=True)
        scores = util.cos_sim(q_emb, self.embeddings)[0]
        top_k = min(top_k, len(self.texts))
        top_idx = torch.argsort(scores, descending=True)[:top_k]
        passages = [self.texts[i] for i in top_idx]
        scores = [scores[i].item() for i in top_idx]
        return passages, scores

retriever = FastRetriever(corpus)


STEP 3: DENSE RETRIEVER

Loading FAST embedding model: sentence-transformers/all-MiniLM-L6-v2
Encoding 104 passages (fast mode)...



Batches:   0%|          | 0/2 [00:00<?, ?it/s]


 Retriever ready! (104 passages)



In [6]:
# ============================================================================
# FAST GENERATOR
# ============================================================================
print("=" * 80)
print("STEP 4: SEQ2SEQ GENERATOR")
print("=" * 80)

class FastGenerator:
    """Faster generator with smaller model"""
    def __init__(self):
        print(f"\nLoading FAST generator: {Config.GENERATOR_MODEL}")
        print("(300MB instead of 3GB - much faster!)\n")

        self.tokenizer = AutoTokenizer.from_pretrained(Config.GENERATOR_MODEL)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(Config.GENERATOR_MODEL)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

        print(f"✓ Generator ready on {self.device}")
        if self.device == "cuda":
            print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print()

    def generate(self, query, passages):
        context = "\n\n".join(passages)
        prompt = f"""Answer based on context.

Question: {query}

Context: {context}

Answer:"""

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=Config.MAX_INPUT_LENGTH,
            truncation=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=Config.MAX_OUTPUT_LENGTH,
                num_beams=2,
                early_stopping=True
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

generator = FastGenerator()

STEP 4: SEQ2SEQ GENERATOR

Loading FAST generator: google/flan-t5-base
(300MB instead of 3GB - much faster!)

✓ Generator ready on cuda
  GPU: Tesla T4



In [7]:
# ============================================================================
# RAG SYSTEM
# ============================================================================
print("=" * 80)
print("STEP 5: RAG SYSTEM")
print("=" * 80)

class RAGSystem:
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator
        print("\n✓ RAG System Ready!\n")

    def answer(self, query):
        passages, scores = self.retriever.retrieve(query)
        answer = self.generator.generate(query, passages)
        return {
            'query': query,
            'answer': answer,
            'retrieved_passages': passages,
            'retrieval_scores': scores
        }

rag = RAGSystem(retriever, generator)

STEP 5: RAG SYSTEM

✓ RAG System Ready!



In [8]:
# ============================================================================
# PROCESS QUERIES
# ============================================================================
print("=" * 80)
print("STEP 6: PROCESSING QUERIES (FAST)")
print("=" * 80)

queries = [json.loads(line)['query'] for line in open(Config.QUERIES_FILE)]
print(f"\n Loaded {len(queries)} queries")
results = []
for query in tqdm(queries, desc="RAG Pipeline"):
    results.append(rag.answer(query))

print(f"\n Processed {len(results)} queries!\n")

STEP 6: PROCESSING QUERIES (FAST)

 Loaded 299 queries


RAG Pipeline: 100%|██████████| 299/299 [02:27<00:00,  2.03it/s]


 Processed 299 queries!



In [9]:
# ============================================================================
# SAVE & DOWNLOAD
# ============================================================================
print("=" * 80)
print("SAVING & DOWNLOADING")
print("=" * 80)

with open(Config.OUTPUT_FILE, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f"\n Saved to: {Config.OUTPUT_FILE}\n")

# Show samples
print("=" * 80)
print("SAMPLE RESULTS")
print("=" * 80 + "\n")

for i, r in enumerate(results[:3], 1):
    print(f"Query {i}: {r['query']}")
    print(f"Answer: {r['answer']}")
    print(f"Top Passage: {r['retrieved_passages'][0][:150]}...")
    print(f"Score: {r['retrieval_scores'][0]:.4f}\n")
    print("-" * 80 + "\n")

# Statistics
ans_lens = [len(r['answer'].split()) for r in results]
avg_scores = [np.mean(r['retrieval_scores']) for r in results]

print("=" * 80)
print("STATISTICS")
print("=" * 80)
print(f"\nAnswer lengths: {np.mean(ans_lens):.1f} words (avg)")
print(f"Retrieval scores: {np.mean(avg_scores):.4f} (avg)")

# Download
print(f"Total processed: {len(results)} queries")
print(f"Results saved to: {Config.OUTPUT_FILE}")
print("Smaller embedding model (all-MiniLM-L6-v2)")
print("Smaller generator (flan-t5-base)")
print("Reduced corpus size (200 passages)")
print("Fewer retrieved passages (3 instead of 5)")
print(" Faster beam search (2 beams)")
print("Shorter context/answers")

SAVING & DOWNLOADING

 Saved to: rag_results.jsonl

SAMPLE RESULTS

Query 1: Explain to me like I’m 5 the theory one can color in each map with 4 colors
Answer: the four color theorem
Top Passage: In this map, the two regions labeled A belong to the same country. If we want those regions to receive the same color, then five colors are required, ...
Score: 0.6306

--------------------------------------------------------------------------------

Query 2: How does sunburn show on black skin?
Answer: swelling
Top Passage: Sunburn will show as varying degrees of skin redness (erythema) and swelling, depending on the extent and severity of sun exposure. Other symptoms can...
Score: 0.6547

--------------------------------------------------------------------------------

Query 3: Explain why some clouds look different
Answer: genus types
Top Passage: In meteorology, a cloud is an aerosol consisting of a visible mass of miniature liquid droplets, ice crystals, or other particles, suspended in 

In [12]:
# ============================================================
#  BASELINE SAMPLE OUTPUTS
# ============================================================
import json
from urllib.parse import quote

# ------------------------------
# 1) Load baseline results
# ------------------------------
print("Loading baseline results...")
baseline_results = []
with open("rag_results.jsonl", "r") as f:
    for line in f:
        baseline_results.append(json.loads(line))

print("Loaded", len(baseline_results), "baseline predictions.\n")

Loading baseline results...
Loaded 299 baseline predictions.



In [13]:
# ------------------------------
# 2) Load corpus used in baseline
# ------------------------------
print("Loading corpus...")
corpus = []
with open("corpus.jsonl", "r") as f:
    for line in f:
        corpus.append(json.loads(line))
print("Loaded", len(corpus), "corpus passages.\n")

Loading corpus...
Loaded 104 corpus passages.



In [14]:
# ------------------------------
# 3) Helper: convert a title → Wikipedia URL
# ------------------------------
def title_to_url(title):
    base = "https://en.wikipedia.org/wiki/"
    return base + quote(title.replace(" ", "_"))

In [15]:
# ------------------------------
# 4) Helper: recover sources for each retrieved passage
# ------------------------------
def recover_sources(passages, corpus):
    recovered = []
    for p in passages:
        snippet = p[:60]  # first 60 chars to match
        for c in corpus:
            if snippet in c["text"]:
                recovered.append(title_to_url(c["source"]))
                break
    return recovered

In [16]:
# ------------------------------
# 5) Print samples in SAME format
# ------------------------------
print("\n================ SAMPLE BASELINE OUTPUTS ================\n")

for idx, r in enumerate(baseline_results[:5]):
    passages = r.get("retrieved_passages", [])
    retrieval_scores = r.get("retrieval_scores", [])

    sources = recover_sources(passages, corpus)
    if len(sources) == 0:
        sources = ["[No source detected]"]

    print("-" * 100)
    print(f"ID:        baseline_{idx}")
    print(f"Query:     {r['query']}")
    print(f"Confidence: N/A (baseline has no confidence estimator)")
    print(f"Sources:   {sources}")
    print(f"Answer:    {r['answer'][:400]}...")
    print()


================ SAMPLE BASELINE OUTPUTS ================

----------------------------------------------------------------------------------------------------
ID:        baseline_0
Query:     Explain to me like I’m 5 the theory one can color in each map with 4 colors
Confidence: N/A (baseline has no confidence estimator)
Sources:   ['https://en.wikipedia.org/wiki/Four_color_theorem', 'https://en.wikipedia.org/wiki/Four_color_theorem', 'https://en.wikipedia.org/wiki/Four_color_theorem']
Answer:    the four color theorem...

----------------------------------------------------------------------------------------------------
ID:        baseline_1
Query:     How does sunburn show on black skin?
Confidence: N/A (baseline has no confidence estimator)
Sources:   ['https://en.wikipedia.org/wiki/Sunburn', 'https://en.wikipedia.org/wiki/Sunburn', 'https://en.wikipedia.org/wiki/Sunburn']
Answer:    swelling...

------------------------------------------------------------------------------------